# 面试题：IVF-PQ 向量检索怎么从零实现，召回率和成本怎样权衡？

本 Notebook 不调用 FAISS/ScaNN/向量数据库，使用 NumPy 手写 train-only 标准化、K-Means、倒排 coarse cells、残差 Product Quantization、ADC 距离、`nprobe`、删除与 Recall@K。重点不是复刻高性能 SIMD kernel，而是把训练、编码、检索和版本合同讲清楚。

合成向量的高召回只证明实现正确；线上还需 GPU/CPU kernel、批查询、内存映射、分片、过滤、动态更新与压测。

In [ ]:
import copy,hashlib,json,math,warnings  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
RNG68=np.random.default_rng(6801)  # 计算并保存当前步骤的中间状态。
def canonical68(x): return json.dumps(x,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha68(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert RNG68 is not None and np.isfinite(np.array([0.])).all()  # 用受控断言验证关键不变量。

## 1. 数据合同与 train-only 预处理

向量是 `[N,D] float32`，ID 唯一，所有值有限。先按时间切 train/index/query：量化器只在 train 拟合；index 可以包含后续已发布向量；query 只评估。均值方差也只能从 train 计算，避免用未来分布调整距离。

是否 L2 normalize 取决于相似度。这里使用标准化后的平方 L2；若业务是 cosine，必须在索引与查询两端使用完全相同的归一化版本。

In [ ]:
centers68=RNG68.normal(size=(8,8)).astype(np.float32)*3  # 计算并保存当前步骤的中间状态。
train68=np.vstack([c+RNG68.normal(scale=.45,size=(30,8)) for c in centers68]).astype(np.float32)  # 计算并保存当前步骤的中间状态。
index68=np.vstack([c+RNG68.normal(scale=.50,size=(20,8)) for c in centers68]).astype(np.float32)  # 计算并保存当前步骤的中间状态。
query68=np.vstack([c+RNG68.normal(scale=.30,size=(3,8)) for c in centers68]).astype(np.float32)  # 计算并保存当前步骤的中间状态。
ids68=np.array([f"v{i:03d}" for i in range(len(index68))])  # 计算并保存当前步骤的中间状态。
mean68=train68.mean(0); std68=train68.std(0); std68=np.where(std68<1e-6,1.,std68).astype(np.float32)  # 计算并保存当前步骤的中间状态。
def transform68(x):  # 定义本节可复用的核心函数。
    x=np.asarray(x,dtype=np.float32)  # 计算并保存当前步骤的中间状态。
    if x.ndim!=2 or x.shape[1]!=8 or not np.isfinite(x).all(): raise ValueError("vector_contract")  # 按当前条件选择后续控制路径。
    return (x-mean68)/std68  # 返回当前分支计算出的结果。
train_z68=transform68(train68); index_z68=transform68(index68); query_z68=transform68(query68)  # 计算并保存当前步骤的中间状态。
assert train_z68.shape==(240,8) and index_z68.shape==(160,8) and query_z68.shape==(24,8)  # 用受控断言验证关键不变量。
assert np.allclose(train_z68.mean(0),0,atol=1e-6) and len(set(ids68))==len(ids68)  # 用受控断言验证关键不变量。
assert mean68.shape==(8,) and std68.shape==(8,) and mean68.dtype==np.float32 and std68.dtype==np.float32  # 用受控断言验证关键不变量。
try: transform68(np.zeros((2,7))); raise AssertionError("wrong dim accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="vector_contract"  # 捕获预期异常并验证失败分支。

## 2. 手写 K-Means：初始化、空簇和确定性

IVF 的 coarse centroid 与 PQ codebook 都依赖 K-Means。这里使用 seeded k-means++ 初始化、广播距离、均值更新和 farthest-point 空簇修复。停止条件是 assignment 不再变化或达到最大迭代。

生产训练通常分批、并行并多次重启；训练样本必须可追溯，空簇策略和浮点精度也要进入制品 manifest。

In [ ]:
def sqdist68(a,b):  # 定义本节可复用的核心函数。
    a=np.asarray(a,np.float32); b=np.asarray(b,np.float32)  # 计算并保存当前步骤的中间状态。
    if a.ndim!=2 or b.ndim!=2 or a.shape[1]!=b.shape[1]: raise ValueError("distance_shape")  # 按当前条件选择后续控制路径。
    return np.maximum((a*a).sum(1)[:,None]+(b*b).sum(1)[None,:]-2*a@b.T,0.)  # 返回当前分支计算出的结果。
def kmeans68(x,k,seed,max_iter=40):  # 定义本节可复用的核心函数。
    if not 1<=k<=len(x): raise ValueError("k_contract")  # 按当前条件选择后续控制路径。
    rng=np.random.default_rng(seed); chosen=[int(rng.integers(len(x)))]  # 计算并保存当前步骤的中间状态。
    for _ in range(1,k):  # 遍历输入元素以累积或检查结果。
        d=sqdist68(x,x[chosen]).min(1); total=float(d.sum())  # 计算并保存当前步骤的中间状态。
        chosen.append(int(rng.choice(len(x),p=d/total)) if total>0 else next(i for i in range(len(x)) if i not in chosen))  # 计算并保存当前步骤的中间状态。
    cent=x[chosen].copy(); old=None  # 计算并保存当前步骤的中间状态。
    for _ in range(max_iter):  # 遍历输入元素以累积或检查结果。
        assign=sqdist68(x,cent).argmin(1)  # 计算并保存当前步骤的中间状态。
        if old is not None and np.array_equal(assign,old): break  # 按当前条件选择后续控制路径。
        old=assign.copy(); nearest=sqdist68(x,cent).min(1)  # 计算并保存当前步骤的中间状态。
        for j in range(k): cent[j]=x[assign==j].mean(0) if np.any(assign==j) else x[int(nearest.argmax())]  # 遍历输入元素以累积或检查结果。
    return cent.astype(np.float32),assign  # 返回当前分支计算出的结果。
toy68=np.array([[0.,0.],[0.,1.],[9.,9.],[9.,10.]],np.float32); tc68,ta68=kmeans68(toy68,2,1)  # 计算并保存当前步骤的中间状态。
assert len(np.unique(ta68))==2 and sqdist68(toy68,tc68).shape==(4,2)  # 用受控断言验证关键不变量。
assert np.array_equal(kmeans68(toy68,2,1)[1],ta68) and np.all(sqdist68(toy68,tc68)>=0)  # 用受控断言验证关键不变量。
try: kmeans68(toy68,5,1); raise AssertionError("too many clusters accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="k_contract"  # 捕获预期异常并验证失败分支。

## 3. Product Quantization 与残差编码

将 8 维残差切成 `M=4` 个 2 维子空间，每个子空间学习 `K=16` 个 codeword。每个向量只保存四个 uint8 code；解码是 codeword 拼接。IVF-PQ 通常量化 `x - coarse_centroid`，比直接量化全局向量误差更小。

子空间必须整除维度；codebook 顺序是二进制协议，任何重训都需要新版本，不能原地替换后继续解释旧 code。

In [ ]:
class ProductQuantizer68:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,m=2,k=16): self.m=m; self.k=k; self.books=None; self.subdim=None  # 定义本节可复用的核心函数。
    def fit(self,residuals,seed=6802):  # 定义本节可复用的核心函数。
        if residuals.shape[1]%self.m: raise ValueError("pq_divisibility")  # 按当前条件选择后续控制路径。
        self.subdim=residuals.shape[1]//self.m; self.books=[]  # 计算并保存当前步骤的中间状态。
        for part in range(self.m): self.books.append(kmeans68(residuals[:,part*self.subdim:(part+1)*self.subdim],self.k,seed+part)[0])  # 遍历输入元素以累积或检查结果。
        self.books=np.stack(self.books); return self  # 计算并保存当前步骤的中间状态。
    def encode(self,residuals):  # 定义本节可复用的核心函数。
        if self.books is None or residuals.ndim!=2 or residuals.shape[1]!=self.m*self.subdim: raise ValueError("pq_not_fitted_or_shape")  # 按当前条件选择后续控制路径。
        return np.stack([sqdist68(residuals[:,i*self.subdim:(i+1)*self.subdim],self.books[i]).argmin(1) for i in range(self.m)],1).astype(np.uint8)  # 返回当前分支计算出的结果。
    def decode(self,codes):  # 定义本节可复用的核心函数。
        if codes.dtype!=np.uint8 or codes.ndim!=2 or codes.shape[1]!=self.m or np.any(codes>=self.k): raise ValueError("code_contract")  # 按当前条件选择后续控制路径。
        return np.concatenate([self.books[i][codes[:,i]] for i in range(self.m)],1)  # 返回当前分支计算出的结果。
pq_probe68=ProductQuantizer68(2,2).fit(np.vstack([np.zeros((3,8)),np.ones((3,8))]).astype(np.float32),3)  # 计算并保存当前步骤的中间状态。
code_probe68=pq_probe68.encode(np.array([[0]*8,[1]*8],np.float32)); dec_probe68=pq_probe68.decode(code_probe68)  # 计算并保存当前步骤的中间状态。
assert code_probe68.shape==(2,2) and code_probe68.dtype==np.uint8 and np.allclose(dec_probe68,[[0]*8,[1]*8])  # 用受控断言验证关键不变量。
try: ProductQuantizer68(3,4).fit(np.zeros((5,8),np.float32)); raise AssertionError("non-divisible PQ accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="pq_divisibility"  # 捕获预期异常并验证失败分支。

## 4. IVF 训练、编码和 posting list

先在 train 学 coarse centroids；每个训练向量减去所属 centroid，再训练 PQ。写入时选择最近 coarse cell，将 `(id, codes, version)` 放进该 posting list。原向量只在教学评估中保留，真实压缩索引可以另存 rerank 向量或完全不存。

写入数据不能反向更新 codebook，否则同一 cell 中 code 语义不一致。增量扩容应新建索引版本并双写迁移。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Entry68: item_id:str; version:int; code:np.ndarray; coarse:int  # 定义承载本节状态与行为的数据结构。
class IVFPQ68:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,nlist=8,m=4,ksub=16): self.nlist=nlist; self.pq=ProductQuantizer68(m,ksub); self.coarse=None; self.lists=defaultdict(list); self.latest={}; self.deleted=set()  # 定义本节可复用的核心函数。
    def fit(self,x):  # 定义本节可复用的核心函数。
        self.coarse,assign=kmeans68(x,self.nlist,6803); residual=x-self.coarse[assign]; self.pq.fit(residual,6804); return self  # 计算并保存当前步骤的中间状态。
    def add(self,ids,x,version=1):  # 定义本节可复用的核心函数。
        if self.coarse is None or len(ids)!=len(x): raise ValueError("index_add_contract")  # 按当前条件选择后续控制路径。
        coarse=sqdist68(x,self.coarse).argmin(1); codes=self.pq.encode(x-self.coarse[coarse])  # 计算并保存当前步骤的中间状态。
        for item,c,code in zip(ids,coarse,codes):  # 遍历输入元素以累积或检查结果。
            if version<=self.latest.get(str(item),0): raise ValueError("stale_vector_version")  # 按当前条件选择后续控制路径。
            self.latest[str(item)]=version; self.deleted.discard(str(item)); self.lists[int(c)].append(Entry68(str(item),version,code.copy(),int(c)))  # 计算并保存当前步骤的中间状态。
    def delete(self,item,version):  # 定义本节可复用的核心函数。
        if version<=self.latest.get(item,0): raise ValueError("stale_vector_version")  # 按当前条件选择后续控制路径。
        self.latest[item]=version; self.deleted.add(item)  # 计算并保存当前步骤的中间状态。
from collections import defaultdict  # 导入本单元所需的依赖。
ivf68=IVFPQ68().fit(train_z68); ivf68.add(ids68,index_z68)  # 计算并保存当前步骤的中间状态。
assert len(ivf68.latest)==160 and sum(map(len,ivf68.lists.values()))==160  # 用受控断言验证关键不变量。
assert ivf68.coarse.shape==(8,8) and ivf68.pq.books.shape==(4,16,2)  # 用受控断言验证关键不变量。
assert all(e.code.dtype==np.uint8 for rows in ivf68.lists.values() for e in rows)  # 用受控断言验证关键不变量。
assert all(e.code.shape==(4,) for rows in ivf68.lists.values() for e in rows)  # 用受控断言验证关键不变量。

## 5. ADC 查询与 `nprobe`

查询先找最近的 `nprobe` 个 coarse cells。对每个 cell，计算 query residual `q-c` 到各子 codeword 的距离表，再按候选 code 查表相加，即 Asymmetric Distance Computation；query 不量化，因此比对称量化更准。

`nprobe` 越大，扫描 code 越多、召回通常越高。过滤条件可能让候选不足，需要 over-fetch 或 filter-aware partition，不能默默返回跨租户结果。

In [ ]:
def search68(index,q,k=5,nprobe=2):  # 定义本节可复用的核心函数。
    q=np.asarray(q,np.float32)  # 计算并保存当前步骤的中间状态。
    if q.shape!=(index.coarse.shape[1],) or not 1<=nprobe<=index.nlist or k<1: raise ValueError("search_contract")  # 按当前条件选择后续控制路径。
    cells=np.argsort(sqdist68(q[None],index.coarse)[0])[:nprobe]; candidates=[]; scanned=0  # 计算并保存当前步骤的中间状态。
    for cell in cells:  # 遍历输入元素以累积或检查结果。
        qr=q-index.coarse[cell]; tables=[]  # 计算并保存当前步骤的中间状态。
        for part in range(index.pq.m): tables.append(sqdist68(qr[part*index.pq.subdim:(part+1)*index.pq.subdim][None],index.pq.books[part])[0])  # 遍历输入元素以累积或检查结果。
        for e in index.lists[int(cell)]:  # 遍历输入元素以累积或检查结果。
            if index.latest.get(e.item_id)!=e.version or e.item_id in index.deleted: continue  # 按当前条件选择后续控制路径。
            distance=sum(float(tables[p][e.code[p]]) for p in range(index.pq.m)); candidates.append((e.item_id,distance)); scanned+=1  # 计算并保存当前步骤的中间状态。
    candidates.sort(key=lambda z:(z[1],z[0])); return candidates[:k],scanned  # 计算并保存当前步骤的中间状态。
near68,scanned68=search68(ivf68,query_z68[0],5,2)  # 计算并保存当前步骤的中间状态。
assert len(near68)==5 and scanned68>=5 and all(np.isfinite(d) for _,d in near68)  # 用受控断言验证关键不变量。
assert [d for _,d in near68]==sorted(d for _,d in near68)  # 用受控断言验证关键不变量。
try: search68(ivf68,query_z68[0],5,0); raise AssertionError("zero nprobe accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="search_contract"  # 捕获预期异常并验证失败分支。

## 6. Recall@K、扫描量与量化误差

exact top-k 用原始 index 向量计算；ANN recall 是近似结果与 exact 集合的交集比例。除了均值，还应看 query 分片、尾部 recall、扫描量和 P99 延迟。这里比较 `nprobe=1/2/8`，并验证全探测并不等于 exact：PQ 距离仍可能改变排序。

量化误差要在独立 validation 上测，不能只汇报训练 residual 的 reconstruction error。

In [ ]:
def exact68(q,k): return [ids68[i] for i in np.argsort(sqdist68(q[None],index_z68)[0])[:k]]  # 定义本节可复用的核心函数。
def recall68(nprobe,k=5):  # 定义本节可复用的核心函数。
    vals=[]; scans=[]  # 计算并保存当前步骤的中间状态。
    for q in query_z68:  # 遍历输入元素以累积或检查结果。
        approx,scan=search68(ivf68,q,k,nprobe); vals.append(len(set(exact68(q,k))&{i for i,_ in approx})/k); scans.append(scan)  # 计算并保存当前步骤的中间状态。
    return float(np.mean(vals)),float(np.mean(scans))  # 返回当前分支计算出的结果。
r1_68,s1_68=recall68(1); r2_68,s2_68=recall68(2); r8_68,s8_68=recall68(8)  # 计算并保存当前步骤的中间状态。
assert 0<=r1_68<=1 and 0<=r8_68<=1 and s1_68<s2_68<=s8_68  # 用受控断言验证关键不变量。
assert r8_68>=r1_68 and r2_68>.75  # 用受控断言验证关键不变量。
assign_val68=sqdist68(query_z68,ivf68.coarse).argmin(1); codes_val68=ivf68.pq.encode(query_z68-ivf68.coarse[assign_val68]); recon_val68=ivf68.coarse[assign_val68]+ivf68.pq.decode(codes_val68)  # 计算并保存当前步骤的中间状态。
assert np.mean((recon_val68-query_z68)**2)<.08 and np.isfinite(recon_val68).all()  # 用受控断言验证关键不变量。

## 7. 删除、旧版本过滤与 compaction

删除写版本化 tombstone；posting 中旧 entry 仍在，但查询根据 `latest` 过滤。compaction 只能在拥有原向量或可重建残差时重写；纯 PQ code 可以复制 live entry，却不能在新 codebook 下重新解释。

线上需监控每个 cell 长度倾斜、tombstone 比例、训练分布漂移和 recall 回归，再决定局部 compact 还是全量重建。

In [ ]:
victim68=near68[0][0]; ivf68.delete(victim68,2)  # 计算并保存当前步骤的中间状态。
after68,_=search68(ivf68,query_z68[0],5,8)  # 计算并保存当前步骤的中间状态。
assert victim68 not in {i for i,_ in after68} and victim68 in ivf68.deleted  # 用受控断言验证关键不变量。
try: ivf68.delete(victim68,1); raise AssertionError("stale delete accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="stale_vector_version"  # 捕获预期异常并验证失败分支。
live_entries68=[e for rows in ivf68.lists.values() for e in rows if ivf68.latest[e.item_id]==e.version and e.item_id not in ivf68.deleted]  # 计算并保存当前步骤的中间状态。
assert len(live_entries68)==159 and len({e.item_id for e in live_entries68})==159  # 用受控断言验证关键不变量。

## 8. 制品绑定与发布接口

需要绑定 mean/std、coarse centroids、PQ books、posting codes、ID/version/tombstone、距离类型和 `nprobe/k` 上限。摘要必须覆盖 dtype、shape 和 bytes；仅保存 Python 配置无法证明 codebook 没被替换。

教学 registry 模拟外部签名根。生产应使用不可变对象存储、KMS 签名、版本吊销和双索引灰度。

In [ ]:
def array_digest68(a):  # 定义本节可复用的核心函数。
    v=np.ascontiguousarray(a); return sha68(str(v.dtype).encode()+canonical68(list(v.shape)).encode()+v.tobytes())  # 计算并保存当前步骤的中间状态。
manifest68={"artifact_id":"ivfpq-demo-v1","metric":"standardized_l2","dim":8,"nlist":8,"m":4,"ksub":16,"mean":array_digest68(mean68),"std":array_digest68(std68),"coarse":array_digest68(ivf68.coarse),"books":array_digest68(ivf68.pq.books),"max_nprobe":8,"tombstones":sorted(ivf68.deleted)}  # 计算并保存当前步骤的中间状态。
TRUST68=MappingProxyType({manifest68["artifact_id"]:sha68(canonical68(manifest68).encode())})  # 计算并保存当前步骤的中间状态。
def load_manifest68(m):  # 定义本节可复用的核心函数。
    actual=copy.deepcopy(m); actual["mean"]=array_digest68(mean68); actual["std"]=array_digest68(std68); actual["coarse"]=array_digest68(ivf68.coarse); actual["books"]=array_digest68(ivf68.pq.books)  # 计算并保存当前步骤的中间状态。
    if TRUST68.get(actual.get("artifact_id"))!=sha68(canonical68(actual).encode()): raise RuntimeError("untrusted_ivfpq")  # 按当前条件选择后续控制路径。
    return MappingProxyType(actual)  # 返回当前分支计算出的结果。
pub68=load_manifest68(manifest68)  # 计算并保存当前步骤的中间状态。
assert pub68["dim"]==8 and isinstance(TRUST68,MappingProxyType)  # 用受控断言验证关键不变量。
forged68=copy.deepcopy(manifest68); forged68["max_nprobe"]=99  # 计算并保存当前步骤的中间状态。
try: load_manifest68(forged68); raise AssertionError("forged IVF config accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="untrusted_ivfpq"  # 捕获预期异常并验证失败分支。
print({"recall@nprobe1":round(r1_68,3),"recall@nprobe2":round(r2_68,3),"scanned2":round(s2_68,1)})  # 执行当前语句以推进本节示例。

## 9. 复杂度、追问与来源

内存约为 `N*M*code_bytes + centroids + posting IDs`；查询扫描约 `nprobe * 平均 cell 长度 * M` 次查表。面试追问通常包括 OPQ、rerank、过滤、在线插入、cell 倾斜、GPU batch 和重建策略。回答时要把“量化召回损失”和“候选预算延迟”分开归因。

- Jégou et al., [Product Quantization for Nearest Neighbor Search](https://lear.inrialpes.fr/pubs/2011/JDS11/jegou_searching_with_quantization.pdf), TPAMI 2011。
- FAISS Research, [Billion-scale similarity search with GPUs](https://arxiv.org/abs/1702.08734)，用于理解生产优化而非本例调用。
- Sivic & Zisserman, [Video Google](https://www.robots.ox.ac.uk/~vgg/publications/2003/Sivic03/sivic03.pdf)，倒排量化思想背景。